# Fair combat playground

This notebook exercises the unified `sts_sim.RunEnv` API. One environment owns the run, exposes a visibility-safe observation for the active screen, enumerates the single legal-action list, and applies those same actions with `step()`. Full state and snapshots are available explicitly for debugging and restoration, but are not policy inputs.

From the repository's `simulator/` directory, start it with:

```bash
uv sync --project python --reinstall-package sts-sim
uv run --project python jupyter lab python/notebooks/fair_combat_playground.ipynb
```

In [1]:
from __future__ import annotations

from random import Random

from sts_sim import Card, FairCombatObservation, Potion, Relic, RunEnv
from sts_sim.notebook import action_label, format_actions, show_actions, show_decision


In [7]:
env = RunEnv.new_ironclad(seed="test", ascension=0)

print(env.observation())
actions = env.legal_actions()
print(actions[0])

env.step(actions[0])

print(env.observation())
actions = env.legal_actions()
print(actions)

env.step(actions[0])

print(env.observation())
actions = env.legal_actions()
print(actions)

env.step(actions[0])

print(env.observation())
actions = env.legal_actions()
print(actions)

env.step(actions[0])

print(env.observation())
actions = env.legal_actions()
print(actions)

env.step(actions[0])

print(env.observation())
actions = env.legal_actions()
print(actions)



Event | phase=event
Act 1 | Floor 0 | Ascension 0 | HP 80/80 | Gold 99
--------------------------------------------------------------------------------
Event: Neow
Choices:
  [0] Talk

Inventory
--------------------------------------------------------------------------------
Deck (10): Strike R x5, Defend R x4, Bash
Relics (1): Burning Blood
Potions (0/3): none
event_choose(option=0, revision=0)
Event | phase=event
Act 1 | Floor 0 | Ascension 0 | HP 80/80 | Gold 99
--------------------------------------------------------------------------------
Event: Neow
Choices:
  [0] choose a colorless card to obtain
  [1] enemies in your next three combats have 1 hp
  [2] lose 8 max hp obtain a random rare relic
  [3] lose your starting relic obtain a random boss relic

Inventory
--------------------------------------------------------------------------------
Deck (10): Strike R x5, Defend R x4, Bash
Relics (1): Burning Blood
Potions (0/3): none
(Action(event_choose(option=0, revision=1)), Action(

## Helpers

`sts_sim.notebook` provides a reusable plain-text formatter for combat and run-level screens, with readable labels for decision-local legal actions. Details come only from the fair observation, so the helpers cannot inspect hidden state.

In [ ]:
# `show_decision(decision)` prints state; `action_label(decision, action)`
# gives a stable readable label for any action from that same decision.
# For legal actions alone, use `show_actions(decision)` or `format_actions(decision)`.


## Inspect the fixture

`decision()` atomically returns the fair observation, its legal actions, and a revision token. Always submit an action from that same decision.

In [ ]:
env = RunEnv.combat_fixture()
decision = env.decision()
assert isinstance(decision.observation, FairCombatObservation)
show_decision(decision)


The observation is a tree of frozen, typed dataclasses, so tab completion and direct field access work naturally.

In [ ]:
print(decision.observation)


## Add typed content for experiments

These helpers are explicit debug mutations. They use the canonical `Card`, `Relic`, and `Potion` enums, apply modeled acquisition effects, and invalidate actions from the previous revision.

In [ ]:
debug_env = RunEnv.new_ironclad(seed="typeddebug")
debug_env.add_card(Card.INFLAME)
debug_env.add_relic(Relic.INK_BOTTLE)
debug_env.add_potion(Potion.FIRE)
print(debug_env.observation())
show_actions(debug_env.decision())


## Take an action and compare a branch

Clone before stepping when you want to compare alternatives. Here the original branch plays Defend while the clone remains at the prior decision.

In [ ]:
unchanged_branch = env.clone()

defend = next(
    action
    for action in decision.actions
    if action.kind == "play_hand_slot"
    and action.hand_slot is not None
    and next(
        visible.card.content_key
        for visible in decision.observation.hand
        if visible.slot == action.hand_slot
    ) == "Defend_R"
)
result = env.step(defend)

print("stepped branch")
show_decision(result.decision)

print("\nunchanged clone")
show_decision(unchanged_branch.decision())


## Run a tiny seeded random policy

This is intentionally not a good player—it is just a compact example of the decision/step loop. The Python RNG picks among already enumerated public actions; simulator legality and transitions remain in Rust.

In [ ]:
random_env = RunEnv.combat_fixture()
policy_rng = Random(7)

for step_index in range(20):
    current = random_env.decision()
    if not current.actions:
        print("no further legal actions")
        break
    action = policy_rng.choice(current.actions)
    print(f"{step_index:02}: {action_label(current, action)}")
    outcome = random_env.step(action)
    if outcome.terminal:
        print("combat ended")
        break


## Your experiment

A useful next step is to replace the random choice with a scoring function over `decision.observation` and `decision.actions`. Keep the returned `Action` intact rather than reconstructing one from positional assumptions.

In [ ]:
experiment_env = RunEnv.combat_fixture()
experiment_decision = experiment_env.decision()

# Pick or score one of experiment_decision.actions here.
for action in experiment_decision.actions:
    print(action_label(experiment_decision, action))
